In [7]:
import numpy as np
import pandas as pd
import arff
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from autofeat import AutoFeatRegressor 
import featuretools as ft


In [3]:
with open("german-credit-R.arff", "r") as file:
    data = arff.load(file)
df = pd.DataFrame(data["data"], columns=[attr[0] for attr in data["attributes"]])
 

In [ ]:
cat_cols = ["Sex", "Housing", "Saving accounts", "Checking account", "Purpose","Risk"]
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,67,male,2,own,None,little,1169,6,radio/TV,good
1,22,female,2,own,little,moderate,5951,48,radio/TV,bad
2,49,male,1,own,little,None,2096,12,education,good
3,45,male,2,free,little,little,7882,42,furniture/equipment,good
4,53,male,2,free,little,little,4870,24,car,bad
...,...,...,...,...,...,...,...,...,...,...
995,31,female,1,own,little,None,1736,12,furniture/equipment,good
996,40,male,3,own,little,little,3857,30,car,good
997,38,male,2,own,little,None,804,12,radio/TV,good
998,23,male,2,free,little,little,1845,45,radio/TV,bad


In [ ]:
X = df.drop(columns=["Risk"])  
y = df["Risk"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Précision : {accuracy_score(y_test, y_pred):.2f}")
print(classification_report(y_test, y_pred))
print("Matrice de confusion :\n", confusion_matrix(y_test, y_pred))

Précision : 0.76
              precision    recall  f1-score   support

           0       0.65      0.37      0.47        59
           1       0.78      0.91      0.84       141

    accuracy                           0.76       200
   macro avg       0.71      0.64      0.66       200
weighted avg       0.74      0.76      0.73       200

Matrice de confusion :
 [[ 22  37]
 [ 12 129]]


In [12]:
#https://medium.com/@boukamchahamdi/autofeat-automating-feature-engineering-with-python-f22ec23265a9
af = AutoFeatRegressor( feateng_steps=2,n_jobs=-1)  

X_train_af = af.fit_transform(X_train, y_train)
X_test_af = af.transform(X_test)
X_train_af.head()
print(f"Nombre de nouvelles features créées : {X_train_af.shape[1] - X_train.shape[1]}")

model_af = LogisticRegression()
model_af.fit(X_train_af, y_train)
y_pred_af = model_af.predict(X_test_af)

print("\n Performance APRÈS AutoFeat")
print(f"Précision : {accuracy_score(y_test, y_pred_af):.2f}")
print(classification_report(y_test, y_pred_af))
print("Matrice de confusion :\n", confusion_matrix(y_test, y_pred_af))

c:\Users\H P\git\projetTutore\venv\lib\site-packages\autofeat\featsel.py:270: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  if np.max(np.abs(correlations[c].ravel()[:i])) < 0.9:
c:\Users\H P\git\projetTutore\venv\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Nombre de nouvelles features créées : 13

 Performance APRÈS AutoFeat
Précision : 0.77
              precision    recall  f1-score   support

           0       0.69      0.37      0.48        59
           1       0.78      0.93      0.85       141

    accuracy                           0.77       200
   macro avg       0.73      0.65      0.67       200
weighted avg       0.75      0.77      0.74       200

Matrice de confusion :
 [[ 22  37]
 [ 10 131]]


In [ ]:
#https://www.analyticsvidhya.com/blog/2018/08/guide-automated-feature-engineering-featuretools-python/
#https://ranasinghiitkgp.medium.com/feature-engineering-using-featuretools-with-code-10f8c83e5f68
#Avec featuretools

df_ft = df.reset_index().rename(columns={'index': 'id'})
es = ft.EntitySet(id="dataset")

es = es.add_dataframe(
    dataframe_name="df_ft",  
    dataframe=df_ft,
    index="id"  
)

#création d'une entité (table) job
es.normalize_entity(
    base_dataframe_name="df_ft",
    new_dataframe_name="job_table",
    index="Job"
)

es.normalize_entity(
    base_dataframe_name="df_ft",
    new_dataframe_name="housing_table",
    index="Housing"
)

es.normalize_entity(
    base_dataframe_name="df_ft",
    new_dataframe_name="purpose_table",
    index="Purpose"
)

feature_matrix, feature_names = ft.dfs(
    entityset=es, 
    target_entity ="df_ft",  
    max_depth=4,
    
)

print(f"Nombre de nouvelles features créées : {feature_matrix.shape[1] - X.shape[1]}")



Nombre de nouvelles features créées : 1


c:\Users\H P\git\projetTutore\venv\lib\site-packages\featuretools\synthesis\deep_feature_synthesis.py:169: UserWarning: Only one dataframe in entityset, changing max_depth to 1 since deeper features cannot be created
  warnings.warn(


In [22]:
print(es)

Entityset: dataset
  DataFrames:
    df_ft [Rows: 1000, Columns: 11]
  Relationships:
    No relationships


In [ ]:
X_train_ft, X_test_ft, y_train_ft, y_test_ft = train_test_split(feature_matrix, y, test_size=0.2, random_state=42)
model = LogisticRegression()
model.fit(X_train_ft, y_train_ft)
y_pred_ft = model.predict(X_test_ft)

print("\n Performance APRÈS AutoFeat")
print(f"Précision : {accuracy_score(y_test_ft, y_pred_ft):.2f}")
print(classification_report(y_test_ft, y_pred_ft))
print("Matrice de confusion :\n", confusion_matrix(y_test_ft, y_pred_ft))
